In [1]:
# CELL 1: Installations and Imports
# ---------------------------------
!pip install -q datasets transformers sentencepiece accelerate evaluate sumy rouge-score nltk

import os
import nltk
import torch
import json # For saving results
from datasets import load_dataset
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline
)
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
import time
import evaluate # Use the evaluate library

# Download NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    print("Downloading NLTK punkt tokenizer...")
    nltk.download('punkt', quiet=True) # Use quiet=True for less verbose output in Kaggle logs

print("Libraries installed and imported.")

Libraries installed and imported.


In [2]:
# CELL 2: Configuration and Setup
# -------------------------------

# --- Configuration ---
# Model choices
T5_MODEL_NAME = "t5-small" # Keep small for Kaggle resource limits. Consider t5-base if GPU allows.

# Fine-tuning parameters (Adjust based on Kaggle GPU limits - P100/T4 have ~16GB VRAM)
TRAIN_MODEL = True # Set to False to skip fine-tuning (e.g., for inference only run)
NUM_TRAIN_EPOCHS = 3 # Keep low for Kaggle time limits (e.g., 1-3)
BATCH_SIZE = 4       # Start low (e.g., 4 or 8 for t5-small). Reduce if OOM error occurs.
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

# Dataset sampling (Use smaller subsets for faster runs on Kaggle)
# Set to None to use full dataset (may exceed time limits)
NUM_TRAIN_SAMPLES = 10000 # Example: Use 10k samples for training
NUM_EVAL_SAMPLES = 10000  # Example: Use 1k samples for validation
NUM_TEST_SAMPLES = 500    # Example: Use 500 samples for final testing

# Output Paths (Use /kaggle/working/ for persistence after commit)
BASE_OUTPUT_DIR = "/kaggle/working/"
MODEL_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "t5-cnn-dailymail-finetuned")
LOGGING_DIR = os.path.join(BASE_OUTPUT_DIR, "logs")
RESULTS_FILE = os.path.join(BASE_OUTPUT_DIR, "evaluation_results.json")
SAMPLE_SUMMARIES_FILE = os.path.join(BASE_OUTPUT_DIR, "sample_summaries.txt")

# Ensure output directories exist
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

# Set device (Leverages Kaggle GPU if enabled in Notebook Settings -> Accelerator)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
# Check GPU details if available
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Memory Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

Using device: cuda
Tesla P100-PCIE-16GB
Memory Allocated: 0.0 GB
Memory Cached:    0.0 GB


In [3]:
# CELL 3: Load Dataset
# --------------------

print("Loading CNN/Daily Mail dataset...")
# This will download (if needed) and cache the dataset.
# Ensure internet is enabled in Kaggle settings for the first run.
try:
    dataset = load_dataset("cnn_dailymail", "3.0.0")
    print("Dataset loaded successfully.")
    print(dataset)
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please ensure internet is enabled in the Kaggle notebook settings.")
    # Optional: Add fallback to load from a pre-downloaded Kaggle dataset if available
    # dataset = load_from_disk("/kaggle/input/cnn-dailymail-dataset-path")
    raise e # Stop execution if dataset fails to load

Loading CNN/Daily Mail dataset...
Dataset loaded successfully.
DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [4]:
# CELL 4: Data Exploration & Preprocessing
# ---------------------------------------

print("\n--- Data Sample ---")
# Check if dataset was loaded and has train split
if 'train' in dataset and len(dataset['train']) > 0:
    print("Article:", dataset['train'][0]['article'][:500] + "...")
    print("Highlights:", dataset['train'][0]['highlights'])
else:
    print("Dataset 'train' split not found or is empty.")
    # Handle error or exit if necessary

# --- Preprocessing for T5 ---
tokenizer = T5Tokenizer.from_pretrained(T5_MODEL_NAME)
PREFIX = "summarize: "

def preprocess_function_t5(examples):
    inputs = [PREFIX + doc for doc in examples['article']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length")

    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples['highlights'], max_length=MAX_TARGET_LENGTH, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    # Map -100 to padding tokens in labels, required by T5 training
    # model_inputs["labels"] = [
    #     [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    # ] # DataCollatorForSeq2Seq handles this automatically if labels are passed normally
    return model_inputs

print("\nPreprocessing dataset for T5...")
# Use smaller subsets defined in config for faster processing on Kaggle
tokenized_datasets = dataset.map(
    preprocess_function_t5,
    batched=True,
    # remove_columns=['article', 'highlights', 'id'] # Keep original columns for TextRank & Eval
)
print("T5 Preprocessing complete.")

# Prepare subsets for training, evaluation, testing
train_dataset = tokenized_datasets["train"].shuffle(seed=42)
if NUM_TRAIN_SAMPLES:
    train_dataset = train_dataset.select(range(NUM_TRAIN_SAMPLES))

eval_dataset = tokenized_datasets["validation"].shuffle(seed=42)
if NUM_EVAL_SAMPLES:
    eval_dataset = eval_dataset.select(range(NUM_EVAL_SAMPLES))

# Need separate tokenized and original test sets
test_dataset_tokenized = tokenized_datasets["test"]
if NUM_TEST_SAMPLES:
    test_dataset_tokenized = test_dataset_tokenized.select(range(NUM_TEST_SAMPLES))

test_dataset_original = dataset["test"]
if NUM_TEST_SAMPLES:
    test_dataset_original = test_dataset_original.select(range(NUM_TEST_SAMPLES))


print(f"Using {len(train_dataset)} samples for training.")
print(f"Using {len(eval_dataset)} samples for validation.")
print(f"Using {len(test_dataset_original)} samples for testing.")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565



--- Data Sample ---
Article: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s...
Highlights: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .

Preprocessing dataset for T5...
T5 Preprocessing complete.
Using 10000 samples for training.
Using 10000 samples for validation.
Using 500 samples for testing.


In [5]:
# CELL 5: Extractive Summarization (TextRank)
# -------------------------------------------

def summarize_textrank(text, sentences_count=3):
    parser = PlaintextParser.from_string(text, SumyTokenizer("english"))
    summarizer = TextRankSummarizer()
    summary_sentences = summarizer(parser.document, sentences_count=sentences_count)
    return " ".join([str(sentence) for sentence in summary_sentences])

print("\n--- Generating TextRank Summaries ---")
textrank_summaries = []
textrank_times = []
test_size = len(test_dataset_original)

for i, example in enumerate(test_dataset_original):
    start_time = time.time()
    try:
        summary = summarize_textrank(example['article'], sentences_count=3) # Standard 3 sentences
    except Exception as e:
        print(f"Warning: TextRank failed for sample {i}. Error: {e}. Using empty summary.")
        summary = "" # Handle potential errors in summarization
    end_time = time.time()
    textrank_summaries.append(summary)
    textrank_times.append(end_time - start_time)
    if (i + 1) % (max(1, test_size // 10)) == 0:
         print(f"Generated TextRank summary {i+1}/{test_size}")

avg_textrank_time = sum(textrank_times) / len(textrank_times) if textrank_times else 0
print(f"Finished TextRank. Average Generation Time: {avg_textrank_time:.4f} seconds")

# Example TextRank output
if test_size > 0:
    print("\nSample TextRank Summary:")
    print("Original Article (start):", test_dataset_original[0]['article'][:300] + "...")
    print("Reference Summary:", test_dataset_original[0]['highlights'])
    print("TextRank Summary:", textrank_summaries[0])
else:
    print("No test samples available to show TextRank example.")


--- Generating TextRank Summaries ---
Generated TextRank summary 50/500
Generated TextRank summary 100/500
Generated TextRank summary 150/500
Generated TextRank summary 200/500
Generated TextRank summary 250/500
Generated TextRank summary 300/500
Generated TextRank summary 350/500
Generated TextRank summary 400/500
Generated TextRank summary 450/500
Generated TextRank summary 500/500
Finished TextRank. Average Generation Time: 0.0253 seconds

Sample TextRank Summary:
Original Article (start): (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the cou...
Reference Summary: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open t

In [6]:
# CELL 6: Abstractive Summarization (T5 Fine-tuning)
# -------------------------------------------------

if TRAIN_MODEL:
    print("\n--- Initializing T5 model for Fine-tuning ---")
    print("--- Starting T5 fine-tuning... ---")
    try:
        # Start Training
        print(">>> Calling trainer.train()... This might take a moment to initialize <<<") # ADD THIS
        train_result = trainer.train()
        print("--- Fine-tuning finished. ---")
        # ... rest of the try block
    except Exception as e:
        print(f"An error occurred during training: {e}")
        TRAIN_MODEL = False
    # Load model and move to GPU if available
    model = T5ForConditionalGeneration.from_pretrained(T5_MODEL_NAME).to(DEVICE)

    # Data collator
    # Automatically pads inputs and labels, creates decoder_input_ids
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    # Training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir=MODEL_OUTPUT_DIR,
        evaluation_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        weight_decay=WEIGHT_DECAY,
        save_total_limit=2,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        predict_with_generate=True,
        fp16=torch.cuda.is_available(),
        logging_dir=LOGGING_DIR,
        logging_steps=100,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",  # <--- ADD OR MODIFY THIS LINE
        # Optional: Add generation parameters for evaluation step if needed
        # generation_max_length=MAX_TARGET_LENGTH,
        # generation_num_beams=4,
    )
    # Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        # compute_metrics=compute_metrics # Can add ROUGE calculation during eval if desired
    )

    # Start Training
    print("--- Starting T5 fine-tuning... ---")
    try:
        train_result = trainer.train()
        print("--- Fine-tuning finished. ---")

        # Save the best model and tokenizer
        trainer.save_model() # Saves the best model according to load_best_model_at_end
        tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
        print(f"Best model saved to {MODEL_OUTPUT_DIR}")

        # Log some training metrics
        metrics = train_result.metrics
        print("Training Metrics:")
        print(metrics) # Contains train_loss, train_runtime, etc.

    except Exception as e:
        print(f"An error occurred during training: {e}")
        # Consider adding cleanup or alternative loading logic here if training fails
        TRAIN_MODEL = False # Ensure we don't try to use a partially trained model below

    # Clean up GPU memory if needed
    del model
    del trainer
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    time.sleep(5) # Short pause

else:
    print("\n--- Skipping T5 Fine-tuning (TRAIN_MODEL=False) ---")


--- Initializing T5 model for Fine-tuning ---
--- Starting T5 fine-tuning... ---
>>> Calling trainer.train()... This might take a moment to initialize <<<
An error occurred during training: name 'trainer' is not defined


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-6-0e7d09027ee2>:47: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


--- Starting T5 fine-tuning... ---


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,1.214300,1.189166
2,1.176200,1.179654
3,1.160900,1.177793


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


--- Fine-tuning finished. ---
Best model saved to /kaggle/working/t5-cnn-dailymail-finetuned
Training Metrics:
{'train_runtime': 1225.8606, 'train_samples_per_second': 24.473, 'train_steps_per_second': 1.53, 'total_flos': 4060254044160000.0, 'train_loss': 1.3293953125, 'epoch': 3.0}


In [7]:
# CELL 7: Abstractive Summarization (T5 Inference)
# -----------------------------------------------

print("\n--- Loading T5 model for Inference ---")
# Load the fine-tuned model if training was done, otherwise load base/pre-trained
inference_model_path = MODEL_OUTPUT_DIR if os.path.exists(os.path.join(MODEL_OUTPUT_DIR, "pytorch_model.bin")) else T5_MODEL_NAME

print(f"Loading model from: {inference_model_path}")
try:
    # Ensure the model is loaded to the correct device for inference
    model = T5ForConditionalGeneration.from_pretrained(inference_model_path).to(DEVICE)
    # Ensure tokenizer matches the loaded model
    tokenizer = T5Tokenizer.from_pretrained(inference_model_path)
    print("Inference model loaded successfully.")
except Exception as e:
    print(f"Error loading inference model from {inference_model_path}: {e}")
    print("T5 summarization will not be available.")
    model = None # Set model to None to prevent errors later

# --- T5 Inference Function ---
def summarize_t5(text, model, tokenizer):
    if model is None or tokenizer is None:
         return "Error: T5 model not available for inference."
    model.eval() # Set model to evaluation mode
    input_text = PREFIX + text
    # Use the tokenizer associated with the loaded model
    inputs = tokenizer(input_text, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True, padding=True).to(DEVICE)

    summary_ids = model.generate(
        inputs['input_ids'],
        num_beams=4,
        max_length=MAX_TARGET_LENGTH + 2, # Account for special tokens
        min_length=30,
        length_penalty=2.0,
        early_stopping=True
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# --- Generate T5 Summaries ---
print("\n--- Generating T5 Summaries ---")
t5_summaries = []
t5_times = []
test_size = len(test_dataset_original)

# Only proceed if model loaded successfully
if model is not None:
    # Use torch.no_grad() for inference to save memory and speed up
    with torch.no_grad():
        for i, example in enumerate(test_dataset_original):
            start_time = time.time()
            try:
                summary = summarize_t5(example['article'], model, tokenizer)
            except Exception as e:
                print(f"Warning: T5 inference failed for sample {i}. Error: {e}. Using empty summary.")
                summary = "" # Handle potential errors
            end_time = time.time()
            t5_summaries.append(summary)
            t5_times.append(end_time - start_time)
            if (i + 1) % (max(1, test_size // 10)) == 0:
                 print(f"Generated T5 summary {i+1}/{test_size}")

    avg_t5_time = sum(t5_times) / len(t5_times) if t5_times else 0
    print(f"Finished T5 generation. Average Generation Time: {avg_t5_time:.4f} seconds")

    # Example T5 output
    if test_size > 0:
        print("\nSample T5 Summary:")
        print("Original Article (start):", test_dataset_original[0]['article'][:300] + "...")
        print("Reference Summary:", test_dataset_original[0]['highlights'])
        print("T5 Summary:", t5_summaries[0])
    else:
        print("No test samples available to show T5 example.")

else:
    print("Skipping T5 summary generation as the model failed to load.")
    t5_summaries = ["T5 Model Unavailable"] * test_size # Fill with placeholder if model failed
    avg_t5_time = 0
    
# Clean up GPU memory after inference if needed
del model
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()


--- Loading T5 model for Inference ---
Loading model from: t5-small
Inference model loaded successfully.

--- Generating T5 Summaries ---
Generated T5 summary 50/500
Generated T5 summary 100/500
Generated T5 summary 150/500
Generated T5 summary 200/500
Generated T5 summary 250/500
Generated T5 summary 300/500
Generated T5 summary 350/500
Generated T5 summary 400/500
Generated T5 summary 450/500
Generated T5 summary 500/500
Finished T5 generation. Average Generation Time: 0.7283 seconds

Sample T5 Summary:
Original Article (start): (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the cou...
Reference Summary: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United Stat

In [8]:
# CELL 8: Evaluation using ROUGE
# -----------------------------

print("\n--- Evaluating Summaries using ROUGE ---")
rouge_metric = evaluate.load('rouge')

# Prepare data for ROUGE calculation
reference_summaries = [ex['highlights'] for ex in test_dataset_original]

# Ensure generated summaries list lengths match reference summaries
if len(textrank_summaries) != len(reference_summaries):
    print(f"Warning: TextRank summary count ({len(textrank_summaries)}) does not match reference count ({len(reference_summaries)}). Evaluation might be inaccurate.")
    # Potentially pad or truncate, but better to fix the generation loop
if len(t5_summaries) != len(reference_summaries):
     print(f"Warning: T5 summary count ({len(t5_summaries)}) does not match reference count ({len(reference_summaries)}). Evaluation might be inaccurate.")
     # Add padding if model failed
     if len(t5_summaries) < len(reference_summaries) and "T5 Model Unavailable" in t5_summaries:
         t5_summaries.extend(["T5 Model Unavailable"] * (len(reference_summaries) - len(t5_summaries)))


# Calculate ROUGE for TextRank
print("Calculating ROUGE for TextRank...")
# Handle empty summaries which can cause errors in rouge-score
valid_textrank_preds = [p if p else " " for p in textrank_summaries] # Replace empty string with space
valid_refs = [r if r else " " for r in reference_summaries]
rouge_results_textrank = rouge_metric.compute(
    predictions=valid_textrank_preds,
    references=valid_refs,
    use_stemmer=True,
    rouge_types=["rouge1", "rouge2", "rougeL"]
)
# Scale scores
rouge_results_textrank = {key: value * 100 for key, value in rouge_results_textrank.items()}


# Calculate ROUGE for T5
print("Calculating ROUGE for T5...")
if any("T5 Model Unavailable" in s for s in t5_summaries):
     print("Skipping ROUGE for T5 as model was unavailable.")
     rouge_results_t5 = {"rouge1": 0, "rouge2": 0, "rougeL": 0} # Placeholder scores
else:
    valid_t5_preds = [p if p else " " for p in t5_summaries]
    rouge_results_t5 = rouge_metric.compute(
        predictions=valid_t5_preds,
        references=valid_refs,
        use_stemmer=True,
        rouge_types=["rouge1", "rouge2", "rougeL"]
    )
    rouge_results_t5 = {key: value * 100 for key, value in rouge_results_t5.items()}


print("\n--- Evaluation Results ---")
print(f"{'Metric':<15}{'TextRank':<15}{'T5 (Fine-tuned/Base)':<25}")
print("-" * 55)
print(f"{'ROUGE-1':<15}{rouge_results_textrank.get('rouge1', 0):<15.2f}{rouge_results_t5.get('rouge1', 0):<25.2f}")
print(f"{'ROUGE-2':<15}{rouge_results_textrank.get('rouge2', 0):<15.2f}{rouge_results_t5.get('rouge2', 0):<25.2f}")
print(f"{'ROUGE-L':<15}{rouge_results_textrank.get('rougeL', 0):<15.2f}{rouge_results_t5.get('rougeL', 0):<25.2f}")
print(f"{'Avg Gen Time (s)':<15}{avg_textrank_time:<15.4f}{avg_t5_time:<25.4f}")

# --- Save results to file ---
evaluation_output = {
    "t5_model_used": inference_model_path,
    "num_test_samples": test_size,
    "textrank_rouge": rouge_results_textrank,
    "t5_rouge": rouge_results_t5,
    "avg_textrank_time_sec": avg_textrank_time,
    "avg_t5_time_sec": avg_t5_time
}

try:
    with open(RESULTS_FILE, 'w') as f:
        json.dump(evaluation_output, f, indent=4)
    print(f"\nEvaluation results saved to {RESULTS_FILE}")
except Exception as e:
    print(f"Error saving evaluation results: {e}")


--- Evaluating Summaries using ROUGE ---


Calculating ROUGE for TextRank...
Calculating ROUGE for T5...

--- Evaluation Results ---
Metric         TextRank       T5 (Fine-tuned/Base)     
-------------------------------------------------------
ROUGE-1        22.56          30.88                    
ROUGE-2        6.76           11.55                    
ROUGE-L        15.12          22.25                    
Avg Gen Time (s)0.0253         0.7283                   

Evaluation results saved to /kaggle/working/evaluation_results.json


In [9]:
# CELL 9: Save Sample Summaries (Optional)
# ---------------------------------------

print("\n--- Saving Sample Summaries ---")
try:
    with open(SAMPLE_SUMMARIES_FILE, 'w', encoding='utf-8') as f:
        f.write("Sample Summaries Comparison\n")
        f.write("=" * 30 + "\n\n")
        # Save first 5 samples (or fewer if test set is smaller)
        num_samples_to_save = min(5, test_size)
        for i in range(num_samples_to_save):
            f.write(f"--- SAMPLE {i+1} ---\n")
            f.write(f"ARTICLE (start):\n{test_dataset_original[i]['article'][:500]}...\n\n")
            f.write(f"REFERENCE:\n{reference_summaries[i]}\n\n")
            f.write(f"TEXTRANK:\n{textrank_summaries[i]}\n\n")
            f.write(f"T5:\n{t5_summaries[i]}\n\n")
            f.write("=" * 30 + "\n\n")
    print(f"Sample summaries saved to {SAMPLE_SUMMARIES_FILE}")
except Exception as e:
    print(f"Error saving sample summaries: {e}")

# Exploration of different formats (Cell 6 from original) can be added here if desired,
# generating summaries with different parameters and printing/saving them.


--- Saving Sample Summaries ---
Sample summaries saved to /kaggle/working/sample_summaries.txt


In [10]:
# CELL 10: Final Check and Cleanup
# --------------------------------
print("\n--- Kaggle Notebook Run Finished ---")
print(f"Check the Output section (on the right panel, under Data -> Output -> /kaggle/working) after committing the notebook for:")
print(f"- Fine-tuned model files in: {MODEL_OUTPUT_DIR}")
print(f"- Evaluation results: {RESULTS_FILE}")
print(f"- Sample summaries: {SAMPLE_SUMMARIES_FILE}")
print(f"- Training logs in: {LOGGING_DIR}")

# List saved files in output directory for verification in logs
print("\nFiles in /kaggle/working/:")
!ls /kaggle/working



--- Kaggle Notebook Run Finished ---
Check the Output section (on the right panel, under Data -> Output -> /kaggle/working) after committing the notebook for:
- Fine-tuned model files in: /kaggle/working/t5-cnn-dailymail-finetuned
- Evaluation results: /kaggle/working/evaluation_results.json
- Sample summaries: /kaggle/working/sample_summaries.txt
- Training logs in: /kaggle/working/logs

Files in /kaggle/working/:
evaluation_results.json  logs  sample_summaries.txt  t5-cnn-dailymail-finetuned


In [11]:
import shutil
import os
from IPython.display import FileLink
import time

# --- Configuration ---
directory_to_zip = "/kaggle/working/"
# Name the output zip file (it will be placed in /kaggle/working/ too)
output_zip_filename_base = "/kaggle/working/complete_working_directory"
output_zip_filepath = output_zip_filename_base + ".zip"
# -------------------

print(f"Attempting to zip the entire directory: {directory_to_zip}")
print("This may take a while if the directory is large...")

# Check if the source directory exists
if os.path.isdir(directory_to_zip):
    try:
        # Remove the target zip file if it already exists to avoid errors
        if os.path.exists(output_zip_filepath):
            print(f"Removing existing zip file: {output_zip_filepath}")
            os.remove(output_zip_filepath)

        start_time = time.time()
        # Create the zip archive
        # shutil.make_archive('output_filename_base', 'format', 'directory_to_archive')
        shutil.make_archive(output_zip_filename_base,  # Base name for the zip file
                            'zip',                     # Format
                            directory_to_zip)          # Directory to archive

        end_time = time.time()
        print(f"Directory successfully zipped to: {output_zip_filepath}")
        print(f"Zipping took: {end_time - start_time:.2f} seconds")

        # Check if the zip file was actually created
        if os.path.exists(output_zip_filepath):
            print("\nCreating download link for the zip file...")
            # Display the link
            display(FileLink(output_zip_filepath))
        else:
            print("Error: Zip file was not found after archiving operation.")

    except Exception as e:
        print(f"An error occurred during zipping: {e}")
        print("This might be due to insufficient disk space in /kaggle/working/ or other issues.")
else:
    print(f"Error: Source directory not found at {directory_to_zip}")

# Optional: Check disk space if zipping fails
# print("\nChecking disk space:")
# !df -h /kaggle/working/

Attempting to zip the entire directory: /kaggle/working/
This may take a while if the directory is large...
Directory successfully zipped to: /kaggle/working/complete_working_directory.zip
Zipping took: 84.84 seconds

Creating download link for the zip file...


/kaggle/working/complete_working_directory.zip